<a href="https://colab.research.google.com/github/cbishop4/MSE7530/blob/main/ClassNotebooks/F2026/Lecture8_InClass_20260928.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size=6>In-class Introduction to Curve Fitting

Here, we will begin to use optimization and curve fitting to extract physical parameters from data. We will use a few types of data sets, including a stretched exponential decay and a peak fitting function. Next week, we will start with some X-ray models - I was not able to get the scientific material to catch up to the analysis material.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

!git clone https://github.com/cbishop4/MSE7530.git

<font size=4> **Scipy.optimize**  
SciPy optimize provides functions for minimizing (or maximizing) objective functions, possibly subject to constraints. It includes solvers for nonlinear problems (with support for both local and global optimization algorithms), linear programming, constrained and nonlinear least-squares, root finding, and curve fitting. https://docs.scipy.org/doc/scipy/reference/optimize.html

You should understand the math from lecture for in-class exams, but you don't really need to know it for the exercises today.

We will start with scipy.optimize.curve_fit, which uses non-linear least squares to fit a function f to data.

In [ ]:
from scipy.optimize import curve_fit

## Stretched Exponential  
A stretched exponential function is often used as a phenomenological description of relaxation in disordered systems. Later in the semester, we will return to X-rays, specifically X-ray photon correlation spectroscopy.  

Start with a user-defined function. A stretched exponential function is represented by  
<font size=5>$f(t) = Ae^{-(\frac{t}{\tau_{KWW}})^{\beta}} + C$  
</font>Where $f(t)$ corresponds to some measured property at time $t$,  
$t$ is the time past $t_0$,  
$\tau_{KWW}$ is a characteristic relaxation time that is usually what we are solving for,  
$\beta$ is a stretching exponent that is a measure of how *heterogeneous* the dynamics are (and can also be somewhat nebulous),  
$A$ is the value of the property at $t_0$,  
and $C$ is the value of the property at infinite $t$.

In [ ]:
def KWW(time, A, tau,beta,C):
  ''' A stretched exponential function.
    Args:
      time (float) : time past t_0
      A (float) : value of property at t=0
      tau (float) : characteristic relaxation time
      beta (float) : stretching exponent
      C (float) : value of property at infinite time
    Returns:
      A * np.exp(-(time/tau)**beta) + C (float) : value of property at time
  '''
  return A * np.exp(-(time/tau)**beta) + C

params_in_order = ['A', 'tau', 'beta', 'C'] # will help us later

### Creating a dictionary
We will use a dictionary, a built-in Python data type, to hold the data for how the structure evolves in Polylactic Acid Glasses. Data is from T. Bennin et. al., "Enhanced Segmental Dynamics of Poly(lactic acid) Glasses during Constant Strain Rate Deformation", *Macromolecules* 52 (17): 6428-6437 (2019). https://doi.org/10.1021/acs.macromol.9b01363

In [ ]:
temps = [315, 317, 320, 323]
decays = {}
for T in temps:
  data = pd.read_csv(f'/content/MSE7530/sampledata/{T}K.csv', header=None, names=['t','I'])
  decays[T] = data

We have created a dictionary that holds 4 dataframes as **values**; there is a separate **key** that corresponds to each value. In this case, our keys are integers. They can be any data type, and are often strings.

In [ ]:
print(f'The keys for our dictionary are {decays.keys()}')

You can use a key to access a single dataframe for the temperature that you want.

In [ ]:
decays[315]

### Plotting the data

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4))
for k in decays.keys():
  for a in ax:
    a.plot(decays[k]['t'],decays[k]['I'],'o',label=f'{k}K')
for a in ax:
  a.set_xlabel('Time (s)')
  a.set_ylabel('Intensity')
  a.legend()
ax[1].semilogx()
ax[0].set_title('Linear Scale')
ax[1].set_title('Log X-Scale')


### Fitting the data

#### 320 K

In [ ]:
result = curve_fit(KWW, decays[320]['t'], decays[320]['I'])

In [ ]:
result

In [ ]:
type(result)

In [ ]:
len(result)

Usually, people will separate the result so that it gives two different things (this is identical to above, just different notation)

In [ ]:
popt, pcov = curve_fit(KWW, decays[320]['t'], decays[320]['I'])

pcov is the covariance matrix. Its diagonals provide the variance of the parameter estimate and allow you to compute standard deviation errors; we will do this later. For now, let's focus on popt, which is the optimal parameters.

In [ ]:
print(f'The optimal parameters are {popt}, corresponding to {params_in_order}')

In [ ]:
fig, ax = plt.subplots()
ax.plot(decays[320]['t'],decays[320]['I'],'o',label='Data')
ax.plot(decays[320]['t'],KWW(decays[320]['t'],*popt),label='Fit')
ax.legend()
ax.set_xlabel('Time (s)')
ax.set_ylabel('Intensity')

#### 323 K

<font color='blue'><font size=4> Repeat this for the T = 323 K data.

In [ ]:
popt, pcov = curve_fit(KWW, decays[323]['t'], decays[323]['I'])

pcov is the covariance matrix. Its diagonals provide the variance of the parameter estimate and allow you to compute standard deviation errors; we will do this later. For now, let's focus on popt, which is the optimal parameters.

In [ ]:
print(f'The optimal parameters are {popt}, corresponding to {params_in_order}')

In [ ]:
fig, ax = plt.subplots()
ax.plot(decays[323]['t'],decays[323]['I'],'o',label='Data')
ax.plot(decays[323]['t'],KWW(decays[323]['t'],*popt),label='Fit')
ax.legend()
ax.set_xlabel('Time (s)')
ax.set_ylabel('Intensity')

In [ ]:
# your code here (add extra cells as needed)

What happened? We need to provide bounds for our parameters and/or fix them based on physical intuition. One way we can fix parameters is by re-defining the equation:

### Constraining from the function

In [ ]:
def KWW2(time, tau):
  ''' A stretched exponential function.
    Args:
      time (float) : time past t_0
      A (float) : value of property at t=0
      tau (float) : characteristic relaxation time
      beta (float) : stretching exponent
      C (float) : value of property at infinite time
    Returns:
      A * np.exp(-(time/tau)**beta) + C (float) : value of property at time
  '''
  beta = 0.5
  C = 0.0
  A = 1.0
  return A * np.exp(-(time/tau)**beta) + C

In [ ]:
popt, pcov = curve_fit(KWW2, decays[323]['t'], decays[323]['I'])

pcov is the covariance matrix. Its diagonals provide the variance of the parameter estimate and allow you to compute standard deviation errors; we will do this later. For now, let's focus on popt, which is the optimal parameters.

In [ ]:
print(f'The optimal parameters is {popt}, which was our only fit parameter')

In [ ]:
fig, ax = plt.subplots()
ax.plot(decays[323]['t'],decays[323]['I'],'o',label='Data')
ax.plot(decays[323]['t'],KWW2(decays[323]['t'],*popt),label='Fit')
ax.legend()
ax.set_xlabel('Time (s)')
ax.set_ylabel('Intensity')

#### Solving for all 4 $\tau$

<font color=blue>Run through this for all 4 using the dictionary. We will do this together. Finish with a plot of $\tau$ vs. Temperature.

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(12,4))
plt.subplots_adjust(wspace=0.25)
taus = []
for k in decays.keys():
  popt, pcov = curve_fit(KWW2, decays[k]['t'], decays[k]['I'])
  ax[0].plot(decays[k]['t'],decays[k]['I'],'o',label=f'{k}K')
  ax[0].plot(decays[k]['t'],KWW2(decays[k]['t'],*popt),label=f'Fit {k}K')
  ax[1].plot(decays[k]['t'],decays[k]['I'],'o',label=f'{k}K')
  ax[1].plot(decays[k]['t'],KWW2(decays[k]['t'],*popt),label=f'Fit {k}K')
  taus.append(popt[0])
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('Intensity')
ax[1].set_xlabel('Time (s)')
ax[1].set_ylabel('Intensity')



taus = np.asarray(taus)
ax[2].plot(decays.keys(),np.log10(taus),'o',label=r'$\tau$')
ax[2].set_xlabel('Temperature (K)')
ax[2].set_ylabel('$Log_{10}$ Relaxation Time (s)')
ax[1].semilogx()

for a in ax:
  a.legend()

### Questions/exercises  
1. Calculate the residual sum of squares for each fit. How would you do this? Discuss as a class. Implement in code if you are able, otherwise leave for a home exercise.  
2. There are also alternate ways to constrain the fitting using bounds. We will briefly address this in class, and I will include them in an at-home exercise.

# X-ray Scattering from Liquids and Glasses

### scipy.special

SciPy has many scientific equations pre-loaded to make our lives easier. Take, for example, equation 4.23 in Als-Nielsen & McMorrow for scattering from a sphere of radius r.

<font size=4>$\mathcal{F}(Q) = \frac{1}{V_{p}}\int_0^R\int_0^{2\pi}\int_0^\pi e^{iQrcos~\theta}r^{2}sin~\theta d\theta d\phi dr$  


<font size=4>$ = \frac{1}{V_{p}}\int_0^R 4\pi \frac{sin(Qr)}{Qr}r^2dr$  


<font size=5>$ = 3[\frac{sin(QR) - QRcos(QR)}{Q^{3}R^{3}}] \equiv \frac{3J_{1}(QR)}{QR}$

<font size=4>Where $J_{1}(x)$ is the Bessel function of the first kind. Bessel functions are solutions to Bessel's ordinary differential equation.  They are often used when solving problems in cylindrical and spherical systems. Therefore, they show up often in scientific computing and SciPy has them built in. First,


```
from scipy import special
```



In [ ]:
from scipy import special

Then, we can use the special scipy functions as we do later, when we calculate form factor scattering.

## Scipy.optimize & Scipy.stats
<font size=4>The optimize module of SciPy is one of the most useful ones you will use. It allows fitting data to any user-defined for built-in function. This ranges from least-squares optimization to Global optimization. We will also use the specific linear regression function in scipy.stats. The three functions are:

```
from scipy import stats
from scipy import optimize

# then what we will use

stats.linregress(x, y)
optimize.curve_fit(func, x, y)
optimize.differential_evolution(func, bounds)
```




<font size=4>See the documentation for each:  
The first two should be possible to follow. Both have worked examples; try the examples and become familiar with the principles, even if it's just copy-pasting the example cells.  
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html
https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html

<font size=4>"Differential Evolution" can be more of a black box for now. Briefly read the documentation, and skim the wikipedia page.  
https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.differential_evolution.html
https://en.wikipedia.org/wiki/Differential_evolution


## Plotting scattering from analytical particle morphologies
<font size=4>As described in the textbook, there are only* 3 particle shapes that it is possible to analytically describe form factor scattering: spheres, discs, and rods. Below are the 3 equations entered as user-defined functions, using the Bessel and sine integral functions from the scipy.special module.

<font size=4>For a sphere:  
$\mathcal{F}(Q) = \frac{1}{V_{p}}\int_0^R\int_0^{2\pi}\int_0^\pi e^{iQrcos~\theta}r^{2}sin~\theta d\theta d\phi dr$  


<font size=4>$ = \frac{1}{V_{p}}\int_0^R 4\pi \frac{sin(Qr)}{Qr}r^2dr$  


<font size=5>$ = 3[\frac{sin(QR) - QRcos(QR)}{Q^{3}R^{3}}] \equiv \frac{3J_{1}(QR)}{QR}$

<font size=4> For some reason I am having trouble with the SciPy Bessel function. It does not match what is plotted in the book. Therefore, for the scattering from a sphere, I have just copied the explicit formula in 4.23 rather than using the Bessel.

In [ ]:
# def FQ2_sphere(radius, q): # scipy Bessel function is not reproducing correct result; what's happening?
#   return (3*special.jv(1,q*radius) / (q*radius))**2
def FQ2_sphere(radius, q):
  return (3*((np.sin(q*radius)- q*radius*np.cos(q*radius)))/(q**3 * radius**3))**2
def FQ2_disc(radius, q):
  return 2 * (1 - (special.jv(1,2*q*radius)/(q*radius)) ) / (q**2 * radius**2)
def FQ2_rod(length, q):
  term1 = (2*special.sici(q*length)[0]) / (q * length)
  term2 = (4 * (np.sin(q*length/2))**2) / (q**2 * length**2)
  return term1 - term2 # I just broke this into two terms to avoid a huge return line

First, let's use these to make a similar graph to Figure 4.14 in the textbook.

In [ ]:
dimension = 100.
qrange = np.linspace(0.0001, 0.2,1000)
fig, ax = plt.subplots()

ax.plot(qrange, FQ2_sphere(dimension,qrange),label='Sphere')
ax.plot(qrange, FQ2_disc(dimension,qrange),label='Disc')
ax.plot(qrange, FQ2_rod(dimension,qrange),label='Rod')
ax.legend()
ax.set_xlabel('Q ($A^{-1}$)')
ax.set_ylabel('$|F(Q)|^{2}$')
ax.set_title('Form factor scattering from different shapes')

Note that the above is a bit different than what's in the book; below we use the table to convert the x-axis to $QR_{g}$ like the figure in the book

In [ ]:
dimension = 100.
qrange = np.linspace(0.0001, .8,1000) # make the q-range a bit longer
QRg_sph = qrange*dimension*(3/5)**.5 # note the 3 different formulas for Rg in the table
QRg_disc = qrange*dimension*(0.5)**0.5
QRg_rod = qrange*dimension*(1/12)**0.5
fig, ax = plt.subplots()
ax.plot(QRg_sph, FQ2_sphere(dimension,qrange),label='Sphere')
ax.plot(QRg_disc, FQ2_disc(dimension,qrange),label='Disc')
ax.plot(QRg_rod, FQ2_rod(dimension,qrange),label='Rod')
ax.legend()
ax.set_xlabel('Q$R_{g}$')
ax.set_ylabel('$|F(Q)|^{2}$')
ax.set_xlim(0,10); ax.set_ylim(0,1.1)

## Effect of radius on the form factor for different shapes

In [ ]:
from matplotlib.pyplot import cm
fig, ax = plt.subplots(1,3,figsize=(15,5))
qrange = np.linspace(0.0001, .5,1000)
radii = [50., 100., 150., 200.]
pcol = iter(cm.viridis(np.linspace(0.,.9,len(radii))))
for r in radii:
  cc = next(pcol)
  ax[0].plot(qrange, FQ2_sphere(r,qrange),label=r,color=cc,linewidth=2)
  ax[1].plot(qrange, FQ2_disc(r,qrange),label=r,color=cc,linewidth=2)
  ax[2].plot(qrange, FQ2_rod(r,qrange),label=r,color=cc,linewidth=2)

for a in ax:
  a.semilogy()
  a.legend(title='Dimension (A)')
  a.set_xlabel('Q ($A^{-1}$)')
  a.set_ylabel('$|F(Q)|^{2}$')
ax[0].set_title('Sphere (radius)'); ax[1].set_title('Disc (radius)'); ax[2].set_title('Rod (length)')

A possible question later: for which types of scattering could you fit polydispersity? Is it possible to do so for discs or rods? What answer shows up for polydispersity in the Guinier analysis - is it the mean? Weighted average? Median?

### In-class 1: For a sphere, find the fringe spacing for each particle radius in $Å^{-1}$. What is the relationship?  
The below will allow you to do this for one at a time. Hint: ax.axvline(value) can let you draw lines on the fringe spacings.

In [ ]:
fig, ax = plt.subplots()
qrange = np.linspace(0.0001, .5,1000)
radius = 10.

ax.plot(qrange, FQ2_sphere(radius,qrange),label=radius,linewidth=2)


ax.semilogy()
ax.legend(title='Dimension (A)')
ax.set_xlabel('Q ($A^{-1}$)')
ax.set_ylabel('$|F(Q)|^{2}$')
ax.set_title(f'Form Factor scattering for dilute sphere with radius {radius} $A^-1$')

## Guinier & Porod Analysis

<font size=4> As seen from the above, it is pretty easy to manually inspect the form factor scattering for a sphere and extract the radius by hand. However, it is much harder for discs and rods. Let's pull out the Rgs by Guinier analysis and compare to the dimensions we made the curves with. To do this we will need to:  
1. Change the axes to be appropriate for Guinier analysis  
2. Identify the proper range in which the Guinier approximation holds
3. Linearly fit the data and find the slope; compare with the known Rg

### Guinier analysis

#### Example 1: Sphere with dimension 100.0 Angstroms

Step 1: Calculate "experimental" scattering and transform axes

In [ ]:
# calculate the scattering for the sphere with R = 100.0 Angstroms.
# we will call this the "experimental scattering"
fig, ax = plt.subplots(1,3,figsize=(15,5))
qrange = np.linspace(0.0001, .5,1000)
r = 100.0
experimental_FQ2 = FQ2_sphere(r,qrange)
ax[0].plot(qrange, experimental_FQ2,label=r)
ax[0].set_xlabel('Q ($A^{-1}$)'); ax[0].set_ylabel('$|F(Q)|^{2}$')
# to do Guinier analysis, we need to plot on an ln I vs Q^2 plot
q2 = qrange**2
lnI = np.log(experimental_FQ2) # note that "np.log" is the natural log, while np.log10 is the base 10 log
for a in [ax[1],ax[2]]:
  a.plot(q2,lnI,label=r)
  a.set_xlabel('$Q^{2}~(A^{-2})$'); ax[1].set_ylabel('ln($|F(Q)|^{2}$)')

ax[2].set_xlim(0, 0.005)
ax[0].set_title('Linear plot'); ax[1].set_title('Transformed axes for Guinier analysis')
ax[2].set_title('Zoom-in of plot 2')



Step 2. Identify Guinier range. By visual inspection, it seems clear that by $Q^{2} = 0.002 A^{-2}$, we are well-past a linear regime. What $QR$ does this correspond to? Let's also identify a "safely" linear regime, which we'll say ends at 0.001.

In [ ]:
print(f' Linear approximation ends around {0.001 * r} to {0.002 * r}QR')

Since no explicit value of QR is given in the textbook, this is a decent number for us to see. Keep it in mind for the future and see if it always holds (I'm not sure).

Step 3. Linear Fit.  
Here we will use the scipy.stats linear regression function. There are several different ways you can do linear fits in Python, but this one is good because it gives statistics.

In [ ]:
# import the stats module where linregress is
from scipy import stats

In [ ]:
# first we need to cut our range to the purely linear regime
linear_indices = np.where(q2 < 0.001)
linear_q2 = q2[linear_indices]
linear_lnI = lnI[linear_indices]
plt.plot(linear_q2,linear_lnI,'o')

We can see that this is not exactly linear the whole way through; maybe we want to use a smaller range. Exactly how small you cut it, in reality, depends on your data density.

In [ ]:
lin_result = stats.linregress(linear_q2,linear_lnI)
lin_result

Now we convert to R using the relationship that for a **sphere** the slope of the $ln I$ vs. $Q^{2}$ plot is equal to $-\frac{R^{2}}{5}$

In [ ]:
slope = lin_result[0]
R = np.sqrt(-5 * slope)
R

We got out a value for R that is 107.8 Angstroms, which is roughly the radius that we used to generate the data. Perhaps the disagreement is due to the non-linearity we saw as we got near 0.001; let's try cutting it at 0.0002

Note that the slope is equal to **R**, not **Rg**. To convert the slope to Rg (in the case of an idealized particle) use the formulae provided in table 4.2.

In [ ]:
linear_indices = np.where(q2 < 0.0002)
linear_q2 = q2[linear_indices]
linear_lnI = lnI[linear_indices]
plt.plot(linear_q2,linear_lnI,'o')
lin_result = stats.linregress(linear_q2,linear_lnI)
lin_result
slope = lin_result[0]
R = np.sqrt(-5 * slope)
print(R)

Now R is equal to 101 Angstroms, which is pretty close to our particle radius. Here I will make an equation to automatically return the slope so we can see the effect of cutting that regime.

In [ ]:
def find_slope_sphere(qmax, q_square, ln_I):
  linear_indices = np.where(q_square < qmax)
  linear_q2 = q_square[linear_indices]
  linear_lnI = ln_I[linear_indices]
  lin_result = stats.linregress(linear_q2,linear_lnI)
  slope = lin_result[0]
  rval = lin_result[2]
  R = np.sqrt(-5 * slope)
  return R, rval # this way we also get an estimate of how linear the fit is




In [ ]:
qmaxes = np.linspace(0.00001, 0.002,100)
calc_rad = []
Rvals = []
for qmax in qmaxes:
  R, rval = find_slope_sphere(qmax, q2, lnI)
  calc_rad.append(R)
  Rvals.append(rval)
fig, ax = plt.subplots(2,1,figsize=(5,8))
ax[0].axhline(100.,color='k')
ax[0].plot(qmaxes, calc_rad,'o')
ax[1].plot(qmaxes, Rvals,'o')
ax[1].set_xlabel('Maximum $q^{2}$ value fit')
ax[0].set_ylabel('Particle Radius from Fit')
ax[1].set_ylabel('$R^{2}$ value for fit')
for a in ax:
  a.set_xticks([0.0001,0.001,0.002])
ax[0].text(0.0015,102,'Actual R')


<font size=4>As we can see, the smallest q-region we can fit (with sufficient signal-to-noise, in the case of experimental data) will give us the most accurate value for R.

## Polydispersity
<font size=4> In the textbook, the function for calculating polydispersity is given by  
$I(Q) = \Delta \rho^{2} \int_{0}^{\infty} D(R)V_{p}(R)^{2}|\mathcal{F}(Q,R)|^{2} ~ dR $

<font size=4> We can calculate this by summing multiple form factors. Take that for a sphere:

In [ ]:
mono100 = FQ2_sphere(100.,qrange)
plt.plot(qrange,mono100,label='100 A monodisperse',color='grey')
mono200 = FQ2_sphere(200.,qrange)
plt.plot(qrange,mono200,label='200 A monodisperse',color='black')
bidisperse = (mono100 + mono200)/2
plt.plot(qrange,bidisperse,label='Bidisperse',color='lime',linewidth=2)
plt.legend()
plt.semilogy()

<font size=4> With increasing polydispersity, we very quickly smear out the scattering. Let's do this for 8 particle sizes:

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4))
particle_sizes = np.linspace(50.,100.,8)
scatter = np.zeros(len(qrange))
for p in particle_sizes:
  scatter += FQ2_sphere(p, qrange)
# now the same number of particle sizes, but a larger dispersity
scatter2 = np.zeros(len(qrange))
particle_sizes2 = np.linspace(50.,800.,8)
for p in particle_sizes2:
  scatter2 += FQ2_sphere(p, qrange)

ax[0].plot(qrange,scatter)
ax[1].plot(qrange,scatter2)
for a in ax:
  a.semilogy()
ax[0].set_title('8 particle sizes, 50-100 nm')
ax[1].set_title('8 particle sizes, 50-800 nm')
